<a href="https://colab.research.google.com/github/Nurdaylight/A-Karpathy-repl/blob/main/15_01_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [204]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [205]:
import urllib.request

url = "https://raw.githubusercontent.com/karpathy/makemore/refs/heads/master/names.txt"
filename = 'names.txt'

with urllib.request.urlopen(url) as f:
    data = f.read().decode("utf-8")

words= data.splitlines()

In [206]:
#create mappings from integers into letters
chars= sorted(set(".".join(words)))
itos={i:s for i,s in enumerate(chars)}
stoi={s:i for i,s in enumerate(chars)}

In [207]:
import random
random.shuffle(words)

In [208]:
#Create test and dev sets
lenwords=len(words)
n1=int(0.8*lenwords)
n2=int(0.9*lenwords)

In [209]:
#create function which creates the data
#It must take in the input list of words and output X and Y ready for training
def data_prep(data,context):
  data= ["."*context +word +"." for word in data]
  Y=[stoi[i] for ch in data for i in ch[context:]]
  X=[word[i:i+context] for word in data for i in range(len(word)-context)]
  X=[stoi[i] for x in X for i in x]
  X=torch.tensor(X).view(-1,context)
  Y=torch.tensor(Y)
  return X, Y

In [214]:
context=3
X_tr,Y_tr=data_prep(words,context)

In [225]:
#Parameters
emb_dim=3
hidden_size=100

C=torch.randn(27,emb_dim)

#
a=torch.ones(1,hidden_size)
b=torch.zeros(1,hidden_size)
W1=torch.randn(9,hidden_size)
b1=torch.zeros(hidden_size)



W2=torch.randn(hidden_size,27)*0.1
b2=torch.zeros(27)
params=[C,W1,W2,b1,b2,a,b]
for p in params:
  p.requires_grad = True


In [229]:
for _ in range(10):
  emb=C[X_tr].view(-1,emb_dim*context)
  z=emb@W1+b1
  z=(z-z.mean(0,keepdim=True))/z.std(0,keepdim=True)

  h=torch.tanh(a*z+b)
  logit=h@W2+b2

  loss=F.cross_entropy(logit,Y_tr)

  for p in params:
    p.grad=None
  loss.backward()
  for p in params:
      p.data += -0.1* p.grad
  print(loss)


tensor(3.4754, grad_fn=<NllLossBackward0>)
tensor(3.4406, grad_fn=<NllLossBackward0>)
tensor(3.4083, grad_fn=<NllLossBackward0>)
tensor(3.3781, grad_fn=<NllLossBackward0>)
tensor(3.3500, grad_fn=<NllLossBackward0>)
tensor(3.3238, grad_fn=<NllLossBackward0>)
tensor(3.2994, grad_fn=<NllLossBackward0>)
tensor(3.2767, grad_fn=<NllLossBackward0>)
tensor(3.2555, grad_fn=<NllLossBackward0>)
tensor(3.2360, grad_fn=<NllLossBackward0>)


In [224]:
loss

tensor(3.4317, grad_fn=<NllLossBackward0>)